In [ ]:
!pip install -q chromadb
!pip install -U -q "google-genai"
!pip install nbformat

## RAG con LangChain
Ejecutamos el cuaderno con las funciones y variables necesarias para añadir datos y consultar la base de datos de Chroma utilizando la interfaz vector store de LangChain. 

In [ ]:
%run ./RAG_LangChain.ipynb

---
<h1>Creacion Multi-Agentes</h1>
<br><h3>1º Instalar ADK</h3>

In [ ]:
# Instalar ADK y LiteLLM para soporte multi-model 

!pip install google-adk -q
!pip install litellm -q

print("Instalacion completada.")

<h3>2º Importar librerias</h3>

In [ ]:

import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm #Para el soporte multi-model
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # Para crear mensage 

import warnings
# Ignorar warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Librerias importadas.")

<h3>3º Importar API Keys</h3>

In [ ]:

from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

# --- Comprobar API keys (Opcional) ---
print("API Keys:")
print(f"Google API Key : {'Si' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"OpenAI API Key : {'Si' if os.environ.get('OPENAI_API_KEY') and os.environ['OPENAI_API_KEY'] != 'YOUR_OPENAI_API_KEY' else 'No '}")
print(f"Anthropic API Key : {'Si' if os.environ.get('ANTHROPIC_API_KEY') and os.environ['ANTHROPIC_API_KEY'] != 'YOUR_ANTHROPIC_API_KEY' else 'No '}")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


<h3>4º Definir el modelo de Gemini</h3>

In [ ]:
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"
print("\nEntorno configurado correctamente.")

<h2>Definir el agente principal</h2>
<br><h3>1º Definir el Agente</h3>

In [ ]:
AGENT_MODEL = MODEL_GEMINI_2_0_FLASH #
#Antes de definirlo , me gustaria añadir otra funcion para el filtrado:
def query_col_filtered(query: str, collection: str, release_year=None):
    # Llama a query_col pasando filter=True
    return query_col(query=query, collection=collection, release_year=release_year, filter=True)


cinema_agent = Agent(
    name="cinema_agent",
    model=AGENT_MODEL, # Can be a string for Gemini or a LiteLlm object
    description="Provides information about movie details.",
    instruction="You are a helpful cinema assistant. "
                "When the user asks about movies, actors, or reviews, "
                "you MUST use the 'query_col' tool to fetch the information from the movie database. "
                "If the tool returns an error or empty information , you MUST use the 'query_col_filtered' tool"
                "If the tool still returns an error or empty information , inform the user politely. "
                "If the tool is successful, present the movie information clearly, "
                "including titles, release dates, ratings, and any relevant metadata.",
    tools=[query_col,query_col_filtered], # Pass the function directly
)

print(f"Agent '{cinema_agent.name}' created using model '{AGENT_MODEL}'.")

<h3>2º Crear Runner y Sesion</h3>

In [ ]:
# Setup Session Service and Runner

# --- Session Management ---
# Key Concept: SessionService stores conversation history & state.
# InMemorySessionService is simple, non-persistent storage for this tutorial.
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "cinema_app"
USER_ID = "user_1"
SESSION_ID = "session_001" # Using a fixed ID for simplicity

# Create the specific session where the conversation will happen
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=cinema_agent, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for agent '{runner.agent.name}'.")

<h3>3º Configurar funciones de interaccion del agente</h3>

In [ ]:
#Define Agent Interaction Function

from google.genai import types # For creating message Content/Parts

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  # Key Concept: run_async executes the agent logic and yields Events.
  # We iterate through events to find the final answer.
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):

      # Key Concept: is_final_response() marks the concluding message for the turn.
      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Handle potential errors/escalations
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          # Add more checks here if needed (e.g., specific error codes)
          break # Stop processing events once the final response is found

  print(f"<<< Agent Response: {final_response_text}")

<h3>4º Interactuar con el agente</h3>

In [ ]:
# Run the Initial Conversation

# We need an async function to await our interaction helper
async def run_conversation():
    await call_agent_async("When was Dracula released?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)
    await call_agent_async("Tell me about The Matrix",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)                                       

await run_conversation()

---
<h2>Crear Dos Agentes e implementarlos</h2>
<br><h3>1º Importar las tools para los agentes (añadir a la BD , formatear texto y buscar en WIkipedia)</h3>

In [ ]:
#Importar tools
%run ./tools.ipynb

<h3>2º Definir los Sub-Agentes</h3>

In [ ]:
#  Definir los sub agentes

# --- Searching Agent ---
searching_agent = None
try:
    searching_agent = Agent(
        model = MODEL_GEMINI_2_0_FLASH,
        name="searching_agent",
        instruction=(
            "You are a Movie Collection Agent. Your ONLY task is to store and manage movies, reviews, "
            "If the user do not define any of the tools requiered information , such as a person , just use the tools you have access to and add the information to the DataBase"
            "and people related to movies in the database. "
            "Use the provided tools to add movies, reviews, or people when relevant. "
            "Do not engage in any other conversation or perform unrelated tasks. "
            "Always ask clarifying questions if the user's input is incomplete."
            "Make sure to dont ask any questions or engage more conversation"   
        ),
        description=(
            "Handles adding movies, reviews, and people to the collection using the provided tools."
        ),
        tools=[add_movies_to_collection,add_reviews_to_collection,add_person_to_collection],
    )
    print(f"✅ Agent '{searching_agent.name}' created using model '{searching_agent.model}'.")
except Exception as e:
    print(f"❌ Could not create Movie Collection agent. Check API Key ({MODEL_GEMINI_2_0_FLASH}). Error: {e}")

# --- Text Agent ---
text_agent = None
try:
    text_agent = Agent(
        model = MODEL_GEMINI_2_0_FLASH,
        name="text_agent",
        instruction=(
            "You are a Text Formatting Agent. Your ONLY task is to take user-provided text or structured data "
            "and format it neatly using the 'dicts_to_markdown_dossier_spacy' tool. "
            "Do not perform any other actions or respond in ways unrelated to formatting. "
            "Always ask for clarification if the input is ambiguous."
            "Make sure to dont ask any questions or engage more conversation"  
        ),
        description=(
            "Handles formatting user-provided text or structured data into a clean Markdown dossier "
            "using the 'dicts_to_markdown_dossier_spacy' tool."
        ),tools=[dicts_to_markdown_dossier_spacy],
    )
    print(f"✅ Agent '{text_agent.name}' created using model '{text_agent.model}'.")
except Exception as e:
    print(f"❌ Could not create Text Formatting agent. Check API Key ({MODEL_GEMINI_2_0_FLASH}). Error: {e}")

<h3>3º Definir el Agente Raiz</h3>

In [ ]:
# Definir el agente raiz con los sub agentes

root_agent = None
runner_root = None

if cinema_agent and searching_agent and text_agent:

    root_agent = Agent(
        name="cinema_root_agent",
        model=MODEL_GEMINI_2_0_FLASH,
        description="Main orchestrator for movie queries. Coordinates search, addition, and text formatting.",
        instruction=(
            "You are the Cinema Root Agent. ALWAYS follow these steps EXACTLY:\n"
            "1. Send the user query to 'cinema_agent'.\n"
            "2. If cinema_agent returns EMPTY, NOT FOUND, or any missing-info message, "
            "you MUST delegate to 'searching_agent' using the user query.\n"
            "3. After searching_agent finishes, you MUST immediately call 'cinema_agent' again "
            "using the same query to retrieve the newly stored information.\n"
            "4. When you finally obtain movie information, you MUST send that information to "
            "'text_agent' for formatting.\n"
            "5. Only output the final formatted text.\n"
            "\n"
            "You MUST follow all 5 steps. NEVER skip step 3. NEVER return a response "
            "directly from searching_agent. NEVER ask questions to the user unless REQUIRED."
        ),
        sub_agents=[cinema_agent, searching_agent, text_agent],
    )

    # Crear runner para este agente raíz
    runner_root = Runner(
        agent=root_agent,
        app_name="cinema_app",
        session_service=session_service
    )

    print(f"✅ Root Agent '{root_agent.name}' created with sub-agents: {[sa.name for sa in root_agent.sub_agents]}")
else:
    print("❌ Cannot create root agent because one or more sub-agents are missing.")
    if not cinema_agent: print(" - Cinema Agent is missing.")
    if not searching_agent: print(" - Searching Agent is missing.")
    if not text_agent: print(" - Text Formatting Agent is missing.")


<h3>4º Interactuar con el agente</h3>

In [ ]:
# Interactuar con el Root Agent de Cine
import asyncio
from IPython.display import display, Markdown #visualizar la salida en markdown correctamente

# Verificar que el root_agent existe
if 'root_agent' in globals() and root_agent:
    
    async def run_cinema_conversation():
        print("\n--- Testing Cinema Agent Team Delegation ---")
        
        # Crear sesión independiente para este test
        session_service = InMemorySessionService()
        APP_NAME = "cinema_app_agent_team"
        USER_ID = "user_1_cinema"
        SESSION_ID = "session_001_cinema"
        
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )
        print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

        # Crear runner para el root_agent
        runner_agent_team = Runner(
            agent=root_agent,
            app_name=APP_NAME,
            session_service=session_service
        )
        print(f"Runner created for agent '{root_agent.name}'.")

        # --- Interacciones de prueba ---
        await call_agent_async(
            query="Tell me about the movie Inception.",
            runner=runner_agent_team,
            user_id=USER_ID,
            session_id=SESSION_ID
        )

        await call_agent_async(
            query="Add the movie Spiderman",
            runner=runner_agent_team,
            user_id=USER_ID,
            session_id=SESSION_ID
        )

        await call_agent_async(
            query="Tell me details about the movie Oppenheimer",
            runner=runner_agent_team,
            user_id=USER_ID,
            session_id=SESSION_ID
        )

    # Ejecutar la conversación
    print("Attempting execution using 'await' (notebook async)...")
    await run_cinema_conversation()
    
else:
    print("\n⚠️ Skipping cinema agent team conversation because 'root_agent' is not defined.")
